In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import time

In [ ]:
class RandomDataset(Dataset):
    def __init__(self, num_samples: int):
        self.img = torch.randn(num_samples, 1, 512, 512)
        self.scalar = torch.randn(num_samples, 1)
        self.label = torch.randn(num_samples, 1)

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        image = self.img[idx]
        scalar = self.scalar[idx]
        label = self.label[idx]
        return image, scalar, label

dataset = RandomDataset(1000)
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

In [ ]:
from pynq import get_rails, DataRecorder

rails = get_rails()
recorder = DataRecorder(rails['12V'].power)

# Test CPU

In [ ]:
from CNetPlusScalar import CNetPlusScalar
model = CNetPlusScalar()
model.load_state_dict(torch.load('pre_trained_w.pt'))

In [ ]:
recorder.record(0.1)

In [ ]:
model.eval()
with torch.no_grad():
    inference_time = 0
    cpu_output = []
    recorder.mark()
    for image, scalar, _ in dataloader:
        # Run inference for each image individually
        start_time = time.time()
        pred = model(image, scalar)
        inference_time += time.time() - start_time
        cpu_output.append(pred)
    recorder.mark()

    # Convert list to numpy array
    cpu_output = np.array(cpu_output)

    # calculate average inference time and FPS
    num_images = len(cpu_output)
    avg_inference_time = inference_time / num_images
    fps = num_images / inference_time

    print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))

In [ ]:
recorder.mark()

# Test DPU

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("../vitisai_bitstream/dpu.bit")

In [ ]:
overlay.load_model("zcu104_CNetPlusScalar.xmodel")

In [ ]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn_img = tuple(inputTensors[0].dims)
shapeIn_scalar = tuple(inputTensors[1].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn_img[0])

output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn_img, dtype=np.float32, order="C"), np.empty(shapeIn_scalar, dtype=np.float32, order="C")]
data_img = input_data[0]
data_scalar = input_data[1]

In [ ]:
inference_time = 0
vitisAI_output = []
recorder.mark()
for image, data_scalar, _ in dataloader:
    data = image.permute(0,2,3,1)
    start_time = time.time()
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)
    inference_time += time.time() - start_time
    vitisAI_output.append(output_data[0])
recorder.mark()

# Convert list to numpy array
vitisAI_output = np.array(vitisAI_output)

num_images = len(vitisAI_output)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))

In [ ]:
recorder.stop()

In [ ]:
# Calculate MSE
mse = np.mean((cpu_output - vitisAI_output) ** 2)
print(f"Mean Squared Error between CPU and VitisAI outputs: {mse:.6f}")

In [ ]:
import matplotlib.pyplot as plt

# Get the timestamps of the marks
mark_indices = []
last_mark = 0.0
for i, mark in enumerate(recorder.frame['Mark']):
    if mark > last_mark:
        mark_indices.append(i)
        last_mark = mark

# Extract the power data
power_data = recorder.frame['12V_power']
time_data = recorder.frame.index

# Create the plot
plt.figure(figsize=(12, 6))

# Color the plot based on the marks
plt.plot(time_data[:mark_indices[2]], power_data[:mark_indices[2]], color='blue', label='CPU Inference')
plt.plot(time_data[mark_indices[2]:], power_data[mark_indices[2]:], color='orange', label='FPGA Inference')

# Add labels and title with larger font size
plt.xlabel('Time', fontsize=14)
plt.ylabel('Power (W)', fontsize=14)
plt.title('Power Consumption of CPU and FPGA Inference for MMS Neural Networks', fontsize=12)

# Add legend with larger font size
plt.legend(fontsize=16)

# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Show the plot
plt.show()

In [ ]:
# Extract the power data
mark_data = recorder.frame['Mark']
time_data = recorder.frame.index

# Create the plot
plt.figure(figsize=(12, 6))

# Color the plot based on the marks
plt.plot(time_data, mark_data)

# Show the plot
plt.show()

In [ ]:
max_power = max(power_data[mark_indices[0]:mark_indices[1]])
print(f"CPU maximum power consumption during execution: {max_power} W")
max_power = max(power_data[mark_indices[3]:mark_indices[4]])
print(f"FPGA maximum power consumption during execution: {max_power} W")